# Limpieza de datos

In [1]:
!pip -q install kagglehub pyarrow pandas numpy matplotlib

In [2]:
import json
import re
from pathlib import Path

import kagglehub
import numpy as np
import pandas as pd
import pyarrow as pa
import pyarrow.parquet as pq

OUTPUT_DIR = Path('/content/yelp_cleaning')
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

OUTPUT_PARQUET = OUTPUT_DIR / 'yelp_review_cleaned.parquet'
OUTPUT_REPORT = OUTPUT_DIR / 'cleaning_report.json'

REQUIRED_COLS = ['review_id', 'user_id', 'business_id', 'stars', 'useful', 'funny', 'cool', 'text', 'date']
REQUIRED_NON_NULL = ['review_id', 'user_id', 'business_id', 'text', 'date']

CHUNK_SIZE = 500_000
MIN_DATE = pd.Timestamp('2019-01-01')

MAX_ROWS_IN_MEMORY = 500_000
SORT_BATCH_ROWS = 150_000
WRITE_BATCH_ROWS = 25_000

In [3]:
dataset_path = kagglehub.dataset_download('yelp-dataset/yelp-dataset')
dataset_root = Path(dataset_path)
candidates = list(dataset_root.rglob('yelp_academic_dataset_review.json'))

if not candidates:
    raise FileNotFoundError('No se encontró yelp_academic_dataset_review.json en la descarga de Kaggle.')

review_file = candidates[0]

if not review_file.exists():
    raise FileNotFoundError(f'No existe el archivo: {review_file}')

print('Archivo fuente:', review_file)
print('Salida parquet:', OUTPUT_PARQUET)
print('Filtro fecha mínima:', MIN_DATE.date())

Using Colab cache for faster access to the 'yelp-dataset' dataset.
Archivo fuente: /kaggle/input/yelp-dataset/yelp_academic_dataset_review.json
Salida parquet: /content/yelp_cleaning/yelp_review_cleaned.parquet
Filtro fecha mínima: 2019-01-01


In [4]:
def _upper_ratio(s: str) -> float:
    letters = [ch for ch in s if ch.isalpha()]
    if not letters:
        return 0.0
    upper = sum(ch.isupper() for ch in letters)
    return upper / len(letters)

def _avg_word_len(s: str) -> float:
    words = re.findall(r"[A-Za-zÀ-ÿ0-9']+", s)
    if not words:
        return 0.0
    return sum(len(word) for word in words) / len(words)

def _sentence_count(s: str) -> int:
    if not s or not s.strip():
        return 0
    count = len(re.findall(r'[.!?]+', s))
    return max(count, 1)

def _all_caps_word_ratio(s: str) -> float:
    words = re.findall(r"[A-Za-zÀ-ÿ]+", s)
    if not words:
        return 0.0
    all_caps = 0
    for word in words:
        letters = [ch for ch in word if ch.isalpha()]
        if letters and len(letters) > 1 and all(ch.isupper() for ch in letters):
            all_caps += 1
    return all_caps / len(words)

In [5]:
stats = {
    'rows_input': 0,
    'rows_after_required_cols': 0,
    'rows_drop_null_required': 0,
    'rows_drop_invalid_date': 0,
    'rows_drop_before_2019': 0,
    'rows_drop_duplicate_review_id': 0,
    'rows_output_cleaned': 0,
    'chunks_processed': 0
}

In [6]:
if OUTPUT_PARQUET.exists():
    OUTPUT_PARQUET.unlink()

writer = None
stream_order_offset = 0

reader = pd.read_json(review_file, lines=True, chunksize=CHUNK_SIZE)

for chunk in reader:
    stats['chunks_processed'] += 1
    stats['rows_input'] += len(chunk)

    missing_cols = [c for c in REQUIRED_COLS if c not in chunk.columns]
    if missing_cols:
        raise ValueError(f'Faltan columnas requeridas: {missing_cols}')

    chunk = chunk[REQUIRED_COLS].copy()
    stats['rows_after_required_cols'] += len(chunk)

    before_null = len(chunk)
    chunk = chunk.dropna(subset=REQUIRED_NON_NULL).copy()
    stats['rows_drop_null_required'] += before_null - len(chunk)

    chunk['date_parsed'] = pd.to_datetime(chunk['date'], errors='coerce')
    before_date = len(chunk)
    chunk = chunk[chunk['date_parsed'].notna()].copy()
    stats['rows_drop_invalid_date'] += before_date - len(chunk)

    before_min_date = len(chunk)
    chunk = chunk[chunk['date_parsed'] >= MIN_DATE].copy()
    stats['rows_drop_before_2019'] += before_min_date - len(chunk)

    before_dedup = len(chunk)
    chunk = chunk.drop_duplicates(subset=['review_id'], keep='first').copy()
    stats['rows_drop_duplicate_review_id'] += before_dedup - len(chunk)

    txt = chunk['text'].astype(str)
    chunk['text_len_chars'] = txt.str.len()
    chunk['text_len_words'] = txt.str.split().str.len()
    chunk['line_breaks'] = txt.str.count('\n')
    chunk['has_url'] = txt.str.contains(r'http[s]?://|www\.', regex=True, case=False, na=False).astype('int8')

    chunk['text_normalized'] = (
        txt.str.lower()
          .str.replace(r'[^\w\s]', ' ', regex=True)
          .str.replace(r'\s+', ' ', regex=True)
          .str.strip()
    )
    chunk['upper_ratio'] = txt.apply(_upper_ratio).astype('float32')
    chunk['all_caps_word_pct'] = txt.apply(_all_caps_word_ratio).astype('float32')
    chunk['sentence_count'] = txt.apply(_sentence_count).astype('int32')
    chunk['avg_sentence_len_words'] = (chunk['text_len_words'] / chunk['sentence_count'].replace(0, np.nan)).fillna(0.0).astype('float32')
    chunk['avg_word_len_chars'] = txt.apply(_avg_word_len).astype('float32')
    chunk['numeral_pct'] = txt.apply(
        lambda s: (sum(ch.isdigit() for ch in s) / len(s.replace(' ', ''))) if s and s.replace(' ', '') else 0.0
    ).astype('float32')

    chunk['date_day'] = chunk['date_parsed'].dt.floor('d')
    chunk['hour_bucket'] = chunk['date_parsed'].dt.floor('h')
    chunk['hour_of_day'] = chunk['date_parsed'].dt.hour.astype('int8')
    chunk['day_of_week'] = chunk['date_parsed'].dt.dayofweek.astype('int8')
    chunk['event_ts'] = (chunk['date_parsed'].astype('int64') // 10**9).astype('int64')

    chunk['text_hash'] = pd.util.hash_pandas_object(txt, index=False).astype('uint64')
    chunk['text_norm_hash'] = pd.util.hash_pandas_object(chunk['text_normalized'], index=False).astype('uint64')

    chunk['stream_order'] = np.arange(
        stream_order_offset,
        stream_order_offset + len(chunk),
        dtype=np.int64
    )
    stream_order_offset += len(chunk)

    out_cols = [
        'stream_order',
        'review_id', 'user_id', 'business_id',
        'date', 'date_parsed', 'event_ts', 'date_day', 'hour_bucket', 'hour_of_day', 'day_of_week',
        'stars', 'useful', 'funny', 'cool',
        'text', 'text_normalized', 'text_hash', 'text_norm_hash',
        'text_len_chars', 'text_len_words', 'line_breaks', 'sentence_count', 'avg_sentence_len_words',
        'avg_word_len_chars', 'numeral_pct', 'upper_ratio', 'all_caps_word_pct', 'has_url'
    ]
    cleaned_chunk = chunk[out_cols].copy()

    table = pa.Table.from_pandas(cleaned_chunk, preserve_index=False)
    if writer is None:
        writer = pq.ParquetWriter(str(OUTPUT_PARQUET), table.schema, compression='snappy')
    writer.write_table(table)

    stats['rows_output_cleaned'] += len(cleaned_chunk)
    print(
        f"Chunk {stats['chunks_processed']}: ",
        f"input={len(chunk):,} | output acumulado={stats['rows_output_cleaned']:,}"
    )

if writer is not None:
    writer.close()
else:
    raise RuntimeError('No se procesaron datos. Verifica el archivo de entrada.')

with open(OUTPUT_REPORT, 'w', encoding='utf-8') as f:
    json.dump(stats, f, ensure_ascii=False, indent=2)

print('Limpieza completada.')
print(json.dumps(stats, indent=2, ensure_ascii=False))
print('Reporte:', OUTPUT_REPORT)
print('Parquet limpio:', OUTPUT_PARQUET)

Chunk 1:  input=83,933 | output acumulado=83,933
Chunk 2:  input=133,193 | output acumulado=217,126
Chunk 3:  input=210,739 | output acumulado=427,865
Chunk 4:  input=128,573 | output acumulado=556,438
Chunk 5:  input=113,636 | output acumulado=670,074
Chunk 6:  input=175,913 | output acumulado=845,987
Chunk 7:  input=203,528 | output acumulado=1,049,515
Chunk 8:  input=89,894 | output acumulado=1,139,409
Chunk 9:  input=119,474 | output acumulado=1,258,883
Chunk 10:  input=211,313 | output acumulado=1,470,196
Chunk 11:  input=154,608 | output acumulado=1,624,804
Chunk 12:  input=99,257 | output acumulado=1,724,061
Chunk 13:  input=175,629 | output acumulado=1,899,690
Chunk 14:  input=212,005 | output acumulado=2,111,695
Limpieza completada.
{
  "rows_input": 6990280,
  "rows_after_required_cols": 6990280,
  "rows_drop_null_required": 0,
  "rows_drop_invalid_date": 0,
  "rows_drop_before_2019": 4878585,
  "rows_drop_duplicate_review_id": 0,
  "rows_output_cleaned": 2111695,
  "chunks_p

In [7]:
import gc
import heapq
import shutil

gc.collect()

source_path = OUTPUT_PARQUET
tmp_dir = OUTPUT_DIR / 'sort_tmp_chunks'
sorted_tmp_path = OUTPUT_DIR / f"{OUTPUT_PARQUET.stem}_event_ts_sorted_tmp.parquet"

if tmp_dir.exists():
    shutil.rmtree(tmp_dir)
tmp_dir.mkdir(parents=True, exist_ok=True)

if sorted_tmp_path.exists():
    sorted_tmp_path.unlink()

source_pf = pq.ParquetFile(source_path)
source_columns = source_pf.schema_arrow.names

if 'event_ts' not in source_columns:
    raise KeyError('La columna event_ts no existe en el parquet limpio.')

sort_by = ['event_ts'] + (['review_id'] if 'review_id' in source_columns else [])

print('Fase 1/2: ordenando chunks con pandas...')
chunk_paths = []
for i, batch in enumerate(source_pf.iter_batches(batch_size=SORT_BATCH_ROWS)):
    chunk_df = batch.to_pandas()
    chunk_df['event_ts'] = pd.to_numeric(chunk_df['event_ts'], errors='coerce')
    chunk_df = chunk_df.sort_values(sort_by, kind='mergesort', na_position='last').reset_index(drop=True)

    chunk_path = tmp_dir / f'sorted_chunk_{i:04d}.parquet'
    chunk_df.to_parquet(chunk_path, index=False, compression='snappy')
    chunk_paths.append(chunk_path)

    print(f'  Chunk ordenado {i + 1}: {len(chunk_df):,} filas')
    del chunk_df
    gc.collect()

if not chunk_paths:
    raise RuntimeError('No se encontraron filas para ordenar.')

chunk_count = len(chunk_paths)
merge_batch_rows = max(1_000, (MAX_ROWS_IN_MEMORY - 2 * WRITE_BATCH_ROWS) // chunk_count)
merge_batch_rows = min(merge_batch_rows, SORT_BATCH_ROWS)

max_rows_planned = chunk_count * merge_batch_rows + 2 * WRITE_BATCH_ROWS
if max_rows_planned > MAX_ROWS_IN_MEMORY:
    raise RuntimeError(
        f'Configuración excede el límite de memoria en filas: {max_rows_planned:,} > {MAX_ROWS_IN_MEMORY:,}'
    )

print(
    f'Fase 2/2: merge k-way en streaming (chunks={chunk_count}, '
    f'merge_batch_rows={merge_batch_rows:,}, write_batch_rows={WRITE_BATCH_ROWS:,})'
 )

value_cols = [c for c in source_columns if c != 'stream_order']
output_cols = ['stream_order'] + value_cols
has_review_id = 'review_id' in value_cols

states = []

def _load_next_batch(state):
    try:
        batch = next(state['iter'])
    except StopIteration:
        return False

    df = batch.to_pandas()
    event_key = pd.to_numeric(df['event_ts'], errors='coerce')
    event_key = event_key.fillna(np.iinfo(np.int64).max).astype('int64').to_numpy(copy=False)

    if has_review_id:
        review_key = df['review_id'].fillna('').astype(str).to_numpy(copy=False)
    else:
        review_key = None

    state['keys'] = (event_key, review_key)
    state['values'] = {c: df[c].to_numpy(copy=False) for c in value_cols}
    state['len'] = len(df)
    state['pos'] = 0
    return True

def _current_key(state):
    pos = state['pos']
    event_key, review_key = state['keys']
    if review_key is None:
        return (int(event_key[pos]), '')
    return (int(event_key[pos]), review_key[pos])

for path in chunk_paths:
    st = {
        'iter': pq.ParquetFile(path).iter_batches(batch_size=merge_batch_rows),
        'keys': None,
        'values': None,
        'len': 0,
        'pos': 0,
    }
    if _load_next_batch(st):
        states.append(st)
    else:
        states.append(None)

heap = []
for idx, st in enumerate(states):
    if st is not None:
        heapq.heappush(heap, (_current_key(st), idx))

writer = None
stream_order_counter = 0
out_buffer = []

while heap:
    _, idx = heapq.heappop(heap)
    st = states[idx]
    pos = st['pos']

    row_values = [st['values'][c][pos] for c in value_cols]
    out_buffer.append((stream_order_counter, *row_values))
    stream_order_counter += 1

    st['pos'] += 1
    if st['pos'] >= st['len']:
        if _load_next_batch(st):
            heapq.heappush(heap, (_current_key(st), idx))
    else:
        heapq.heappush(heap, (_current_key(st), idx))

    if len(out_buffer) >= WRITE_BATCH_ROWS:
        out_df = pd.DataFrame.from_records(out_buffer, columns=output_cols)
        table = pa.Table.from_pandas(out_df, preserve_index=False)
        if writer is None:
            writer = pq.ParquetWriter(str(sorted_tmp_path), table.schema, compression='snappy')
        writer.write_table(table)
        out_buffer.clear()
        del out_df
        gc.collect()

if out_buffer:
    out_df = pd.DataFrame.from_records(out_buffer, columns=output_cols)
    table = pa.Table.from_pandas(out_df, preserve_index=False)
    if writer is None:
        writer = pq.ParquetWriter(str(sorted_tmp_path), table.schema, compression='snappy')
    writer.write_table(table)
    out_buffer.clear()
    del out_df

if writer is not None:
    writer.close()
else:
    raise RuntimeError('No se pudo escribir el parquet ordenado.')

sorted_tmp_path.replace(source_path)
shutil.rmtree(tmp_dir, ignore_errors=True)
gc.collect()

print('Parquet reordenado sin cargar todo en RAM:', source_path)
print('Filas totales ordenadas:', f'{stream_order_counter:,}')

parquet_file = pq.ParquetFile(source_path)

cols = [
    'event_ts', 'hour_of_day', 'day_of_week',
    'stars', 'text_len_chars', 'text_len_words', 'sentence_count',
    'avg_sentence_len_words', 'avg_word_len_chars', 'numeral_pct',
    'upper_ratio', 'all_caps_word_pct', 'has_url'
 ]

first_batch = next(
    parquet_file.iter_batches(batch_size=20_000, columns=cols),
    None
 )

if first_batch is None:
    raise RuntimeError('El parquet generado está vacío.')

clean_preview = first_batch.to_pandas()
display(clean_preview.describe().T)

norm_cols = [
    'stream_order', 'event_ts', 'user_id', 'date_parsed', 'text', 'text_normalized',
    'text_hash', 'text_norm_hash'
 ]
norm_batch = next(
    parquet_file.iter_batches(batch_size=5, columns=norm_cols),
    None
 )

if norm_batch is None:
    raise RuntimeError('No hay filas para mostrar en el preview.')

norm_preview = norm_batch.to_pandas()
display(norm_preview)

Fase 1/2: ordenando chunks con pandas...
  Chunk ordenado 1: 150,000 filas
  Chunk ordenado 2: 150,000 filas
  Chunk ordenado 3: 150,000 filas
  Chunk ordenado 4: 150,000 filas
  Chunk ordenado 5: 150,000 filas
  Chunk ordenado 6: 150,000 filas
  Chunk ordenado 7: 150,000 filas
  Chunk ordenado 8: 150,000 filas
  Chunk ordenado 9: 150,000 filas
  Chunk ordenado 10: 150,000 filas
  Chunk ordenado 11: 150,000 filas
  Chunk ordenado 12: 150,000 filas
  Chunk ordenado 13: 150,000 filas
  Chunk ordenado 14: 150,000 filas
  Chunk ordenado 15: 11,695 filas
Fase 2/2: merge k-way en streaming (chunks=15, merge_batch_rows=30,000, write_batch_rows=25,000)
Parquet reordenado sin cargar todo en RAM: /content/yelp_cleaning/yelp_review_cleaned.parquet
Filas totales ordenadas: 2,111,695


,count,mean,std,min,25%,50%,75%,max
event_ts,20000.0,1.546624e+09,176844.421638,1.546301e+09,1.546472e+09,1.546633e+09,1.546768e+09,1.546958e+09
hour_of_day,20000.0,1.224190e+01,8.318482,0.000000e+00,3.000000e+00,1.500000e+01,2.000000e+01,2.300000e+01
day_of_week,20000.0,3.099150e+00,1.987012,0.000000e+00,1.000000e+00,3.000000e+00,5.000000e+00,6.000000e+00
stars,20000.0,3.764350e+00,1.519024,1.000000e+00,3.000000e+00,4.000000e+00,5.000000e+00,5.000000e+00
text_len_chars,20000.0,5.491588e+02,498.887032,3.600000e+01,2.250000e+02,3.940000e+02,7.050000e+02,4.998000e+03
text_len_words,20000.0,1.013764e+02,92.911073,3.000000e+00,4.100000e+01,7.300000e+01,1.300000e+02,9.670000e+02
sentence_count,20000.0,7.796900e+00,6.214023,1.000000e+00,4.000000e+00,6.000000e+00,1.000000e+01,7.200000e+01
avg_sentence_len_words,20000.0,1.330532e+01,7.617899,1.750000e+00,9.400000e+00,1.212903e+01,1.550000e+01,3.310000e+02
avg_word_len_chars,20000.0,4.323844e+00,0.368782,3.000000e+00,4.083333e+00,4.276733e+00,4.505406e+00,8.303030e+00
numeral_pct,20000.0,3.536456e-03,0.007526,0.000000e+00,0.000000e+00,0.000000e+00,4.635442e-03,3.295455e-01


,stream_order,event_ts,user_id,date_parsed,text,text_normalized,text_hash,text_norm_hash
0,0,1546300821,xbJbIcK9SjH19qDKJz5dbw,2019-01-01 00:00:21,this company is a fraud. my laptop was stolen...,this company is a fraud my laptop was stolen f...,6950387208943599872,8130721070853882326
1,1,1546300825,0v-STvwT1JaTZnjtYKt0nQ,2019-01-01 00:00:25,"Incredible cakes, cup cakes and deserts. Perso...",incredible cakes cup cakes and deserts persona...,10354955308172348200,1087651577668249535
2,2,1546300834,l-ulZk9X3_1GHz627e90zw,2019-01-01 00:00:34,It's... Pretty good I guess? Been here a coupl...,it s pretty good i guess been here a couple of...,13593409468193807196,16410284490785545400
3,3,1546300837,S-ozAyU5oOKHt7x9MHICKw,2019-01-01 00:00:37,I think that this place is pretty awesome.Open...,i think that this place is pretty awesome open...,13545834684764994300,15883511512115571656
4,4,1546300847,SswjpMl44u_ea8OllT4ryg,2019-01-01 00:00:47,I had several dishes here. The tacos were a fr...,i had several dishes here the tacos were a fre...,8408016520916585151,800576059418858041


In [8]:
!ls /content/yelp_cleaning -sh

total 1.5G
4.0K cleaning_report.json  1.5G yelp_review_cleaned.parquet
